# 小时级模型特征工程：历史需求特征

目标：在原有日期、小时和天气特征基础上，加入**过去已经观测到的租赁量**，观察是否能改善小时级预测。

注意：这里的滞后特征只使用当前时刻之前的 `cnt`。在真实部署时，系统需要持续保存历史租赁量；因此本实验先用于验证模型改进潜力，不会立即替换网页模型。

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

DATA_PATH = Path('../data/hour.csv')
BASE_FEATURES = [
    'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday',
    'weathersit', 'temp', 'atemp', 'hum', 'windspeed',
]

df = pd.read_csv(DATA_PATH, parse_dates=['dteday'])
df = df.sort_values(['dteday', 'hr']).reset_index(drop=True)
df.shape

(17379, 17)

## 1. 构造滞后特征

- `lag_1`：前一小时的实际租赁量；
- `lag_24`：前一天同一小时的实际租赁量；
- `lag_168`：前一周同一小时的实际租赁量；
- `rolling_mean_24`：过去 24 小时的平均租赁量。

所有特征都先 `shift(1)` 再滚动计算，确保不会把当前小时的目标值泄漏给模型。

In [2]:
feature_df = df.copy()
feature_df['lag_1'] = feature_df['cnt'].shift(1)
feature_df['lag_24'] = feature_df['cnt'].shift(24)
feature_df['lag_168'] = feature_df['cnt'].shift(168)
feature_df['rolling_mean_24'] = feature_df['cnt'].shift(1).rolling(24).mean()

LAG_FEATURES = ['lag_1', 'lag_24', 'lag_168', 'rolling_mean_24']
feature_df = feature_df.dropna(subset=LAG_FEATURES).reset_index(drop=True)
print(f'构造特征后可用样本数：{len(feature_df)}')
feature_df[['dteday', 'hr', 'cnt'] + LAG_FEATURES].head()

构造特征后可用样本数：17211


,dteday,hr,cnt,lag_1,lag_24,lag_168,rolling_mean_24
0,2011-01-08,7,9,2.0,84.0,16.0,63.208333
1,2011-01-08,8,15,9.0,210.0,40.0,60.083333
2,2011-01-08,9,20,15.0,134.0,32.0,51.958333
3,2011-01-08,10,61,20.0,63.0,13.0,47.208333
4,2011-01-08,11,62,61.0,67.0,1.0,47.125000


## 2. 保持时间顺序进行对比

使用相同的随机森林配置，分别训练“基础特征模型”和“加入历史需求特征的模型”。前 80% 用于训练，后 20% 作为最终测试集。

In [3]:
split_index = int(len(feature_df) * 0.8)
train = feature_df.iloc[:split_index]
test = feature_df.iloc[split_index:]

def evaluate(name, features):
    model = RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=1)
    model.fit(train[features], train['cnt'])
    prediction = model.predict(test[features])
    result = {
        '模型': name,
        'MAE': mean_absolute_error(test['cnt'], prediction),
        'RMSE': root_mean_squared_error(test['cnt'], prediction),
        'R²': r2_score(test['cnt'], prediction),
    }
    return model, prediction, result

base_model, base_prediction, base_result = evaluate('基础特征随机森林', BASE_FEATURES)
lag_model, lag_prediction, lag_result = evaluate('加入历史需求特征的随机森林', BASE_FEATURES + LAG_FEATURES)

pd.DataFrame([base_result, lag_result]).set_index('模型').round(3)

,MAE,RMSE,R²
模型,,,
基础特征随机森林,44.751,69.510,0.900
加入历史需求特征的随机森林,33.045,56.583,0.934


## 3. 解释新模型

若加入历史需求特征后指标提升，说明历史租赁量携带了当前天气和日期特征难以完全表达的短期变化信息。

In [4]:
importance = (
    pd.Series(lag_model.feature_importances_, index=BASE_FEATURES + LAG_FEATURES)
    .sort_values(ascending=False)
    .to_frame('特征重要性')
)
importance

,特征重要性
lag_1,0.669969
lag_168,0.133648
lag_24,0.079201
hr,0.063753
workingday,0.010031
rolling_mean_24,0.008788
atemp,0.006245
hum,0.005896
weekday,0.005242
temp,0.004706


## 小结：引入历史需求特征的小时级预测

本阶段在原有日期、小时、天气等基础特征上，加入了历史租赁量特征：

- `lag_1`：前一小时的真实租赁量；
- `lag_24`：前一天同一小时的真实租赁量；
- `lag_168`：前一周同一小时的真实租赁量；
- `rolling_mean_24`：过去 24 小时租赁量的平均值。

### 模型结果

| 模型 | MAE | RMSE | R² |
|---|---:|---:|---:|
| 基础特征随机森林 | 44.751 | 69.510 | 0.900 |
| 加入历史需求特征的随机森林 | **33.045** | **56.583** | **0.934** |

加入历史需求特征后，MAE 降低约 26%，RMSE 降低约 19%，R² 从 0.900 提升到 0.934，说明模型预测效果显著提升。

### 特征重要性

`lag_1` 的重要性最高，约为 0.670，说明当前小时的租赁需求与前一小时需求具有很强的连续性。其次是 `lag_168` 和 `lag_24`，说明共享单车需求还存在明显的周周期性和日周期性。

### 结论与局限

历史需求特征能有效补充日期、天气等基础特征无法完全表达的短期变化信息，因此更适合短期预测。

但该模型依赖过去真实的租赁量数据。实际部署时，需要持续获取并保存历史租赁数据；如果没有真实历史数据，就不能可靠使用该高级模型进行预测。